# Research Paper Summarizer Trials


In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\HP\miniconda3\envs\research-paper-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Data


In [2]:
loader = DirectoryLoader(
    "P:\Generative AI projects\Research-Paper-Summarizer\data",
    glob= "*.pdf",
    loader_cls = PyPDFLoader
)
documents = loader.load()

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, chunk_overlap = 50
)
text_chunks = splitter.split_documents(documents)

In [4]:
len(text_chunks)

1399

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
query_embeddings = embeddings.embed_query("Hello World")
len(query_embeddings)

384

In [4]:
chroma_path = "P:\Generative AI projects\Research-Paper-Summarizer\chroma_db"

if os.path.exists(chroma_path):
    print("🔄 Loading existing Chroma index...")
    vector_db = Chroma(persist_directory = chroma_path, embedding_function = embeddings )

else:
    print("✨ Creating new Chroma index...")
    vector_db = Chroma.from_documents(
        documents=text_chunks,
        embedding=embeddings,
        persist_directory=chroma_path
    )

🔄 Loading existing Chroma index...


C:\Users\HP\AppData\Local\Temp\ipykernel_3944\3643165881.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory = chroma_path, embedding_function = embeddings )


In [5]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    max_tokens = 1000
)

In [ ]:
# message = llm.invoke("Hii")
# message

AIMessage(content='Hi, how can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 37, 'total_tokens': 47, 'completion_time': 0.01411208, 'prompt_time': 0.002079283, 'queue_time': 0.098620451, 'total_time': 0.016191363}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_90c2e79dab', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--e0124ee3-7f02-4a5c-820b-d9a71197ca2a-0', usage_metadata={'input_tokens': 37, 'output_tokens': 10, 'total_tokens': 47})

In [6]:
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

In [28]:
data = retriever.invoke("What are Startup companies")
data

[Document(metadata={'page_label': '2', 'creator': 'Microsoft® Office Word 2007', 'author': 'user', 'moddate': 'D:20150828162218', 'source': 'P:\\Generative AI projects\\Research-Paper-Summarizer\\data\\StartupCompanies-LifeCycleandChallenges.pdf', 'page': 1, 'rgid': 'PB:280007861_AS:267463891877938@1440779769861', 'total_pages': 12, 'creationdate': 'D:20150828162218', 'producer': 'Microsoft® Office Word 2007'}, page_content='Startup Companies: Life Cycle and Challenges \n \nAidin Salamzadeh (Corresponding author) \nFaculty of Entrepreneurship,  University of Tehran,  16th Street, North Kargar  \nAvenue, Tehran, 1439813141, Iran Salamzadeh@ut.ac.ir \nHiroko Kawamorita Kesim \nFaculty of Engineering, Ondokuz Mayıs University, 55200 Atakum -Samsun, \nTurkey hiroko.kawamorita@omu.edu.tr \n \nAbstract \nStartup companies are newly born companies which struggle for existence. These'),
 Document(metadata={'rgid': 'PB:280007861_AS:267463891877938@1440779769861', 'producer': 'Microsoft® Office 

In [ ]:
# for r in data:
#     print(r.page_content)

Startup Companies: Life Cycle and Challenges 
 
Aidin Salamzadeh (Corresponding author) 
Faculty of Entrepreneurship,  University of Tehran,  16th Street, North Kargar  
Avenue, Tehran, 1439813141, Iran Salamzadeh@ut.ac.ir 
Hiroko Kawamorita Kesim 
Faculty of Engineering, Ondokuz Mayıs University, 55200 Atakum -Samsun, 
Turkey hiroko.kawamorita@omu.edu.tr 
 
Abstract 
Startup companies are newly born companies which struggle for existence. These
literature, there are many studies which examined controversial issues in this 
domain (Salamzadeh, 2015b). Amidst this chaos, a challenge arose: what are these 
entities, i.e. startups, and how they turn into companies?  
The work of scholars of management, organization, and entrepreneurship, and 
others who might pursue this challenge, will affect the heavily li fting of applying 
theories to make a clear picture of these entities  (Salamzadeh, 2015 a, b) . Due to
companies which play a significant role in economies - “success stories” (e.g. 

In [7]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """
You are Research Paper Bot specialized in summarizing and answering questions 
based only on the research paper provided.

Use ONLY the retrieved context:
{context}

Rules:
- If the user asks about anything not in the context, reply with: 
  "I don’t have information about that as it's not in the research paper."
- Do NOT answer from your own knowledge.
- Stick strictly to the research paper database.
- Be concise, clear, friendly, and include practical details when possible.
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}')
])

In [8]:
from langchain.chains import create_retrieval_chain
from langchain.chains.history_aware_retriever import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.memory import ConversationBufferMemory

condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a question rewriter for a Research Paper RAG system. "
     "Given the chat history and a follow-up, rewrite it as a standalone search query."),
    ("human", 
     "Chat history:\n{chat_history}\n\n"
     "Follow-up question: {input}\n\n"
     "Standalone query:")
])

histoory_aware_chain = create_history_aware_retriever(
    llm,
    retriever,
    prompt=condense_prompt
)

qa_chain = create_stuff_documents_chain(
    llm, qa_prompt
)

rag_chain = create_retrieval_chain(
    histoory_aware_chain, qa_chain
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

C:\Users\HP\AppData\Local\Temp\ipykernel_3944\2302593084.py:30: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [9]:
def ask_bot(query: str):
    response = rag_chain.invoke({
        "input": query,
        "chat_history": memory.load_memory_variables({})["chat_history"]
    })
    memory.save_context({"input": query}, {"output": response["answer"]})
    return response["answer"]

In [33]:
ask_bot("What are the Lifecycle of startups")

"According to the research paper, the lifecycle of startups is composed of the following stages:\n\n1. Bootstrapping stage\n2. (Unfortunately, the research paper does not list the remaining stages, it only mentions i) Bootstrapping stage and then skips to a mention of environmental elements.)\n\nHowever, it seems that there are missing stages between Bootstrapping stage and Environmental elements.\n\nA figure (Figure 1) in the paper illustrates the lifecycle of startups, which lists the stages. However, the research paper does not elaborate on the stages beyond Bootstrapping.\n\nIf you would like to know more about the specific stages, I'd suggest referring to Figure 1 in the paper for a complete list."

In [35]:
ask_bot("Explain the first stage in detail")

'The first stage in the lifecycle of startups, as mentioned in the research paper, is the Bootstrapping stage. Here are the key points about this stage:\n\n**Bootstrapping Stage:**\n\n* This is the very early stage where the entrepreneur initiates a set of activities.\n* Some scholars consider this stage as the pre-seed stage, while others consider it as the startup stage.\n* The bootstrapping stage is the period between the nascence of a business idea and the moment of sustainable profits.\n* The author defines "startups" as the early stage of any business, venture, or project.\n\nIn essence, the bootstrapping stage is the initial phase where the entrepreneur takes the first steps towards turning their business idea into a reality, without external funding or support.'

In [ ]:
ask_bot("What are some applications of AI in distribution power systems")

"According to the research paper, the applications of AI in distribution power systems include:\n\n* Integration of distributed energy resources into the smart grid using AI and emerging techniques\n* Analysis of individual requirements and essential functions derived from AI applications in distribution system operation\n* Use of metaheuristic algorithms for various applications in power systems (referring to Cai et al. [16])\n\nThese applications aim to provide a systematic overview of AI's capabilities in distribution power system operation."

In [11]:
d = "APPLICATIONS OF AI IN DISTRIBUTION POWER SYSTEMS"
d = d.lower()
d

'applications of ai in distribution power systems'